In [ ]:
from pytrends.request import TrendReq
import pandas as pd

print("Fetching Google Trends data for Asthma (Delhi)...")

pytrends = TrendReq(hl='en-IN', tz=330, retries=3, backoff_factor=2)


pytrends.build_payload(["asthma"], timeframe='today 3-m', geo='IN-DL')


trends_df = pytrends.interest_over_time()


if 'isPartial' in trends_df.columns:
    trends_df = trends_df.drop(columns=['isPartial'])


trends_df.tail()


In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

print("Fetching historical PM2.5 and AQI data for Delhi...")

end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')

lat, lon = 28.6139, 77.2090


url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=pm2_5,us_aqi&timezone=Asia%2FCalcutta&start_date={start_date}&end_date={end_date}"

response = requests.get(url)
data = response.json()

aqi_df = pd.DataFrame(data['hourly'])


aqi_df = aqi_df.dropna()

print(f"Data alignment complete! Fetched {len(aqi_df)} hours of historical data.")
display(aqi_df.tail(10))

In [ ]:
 
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from pytrends.request import TrendReq

print("--- INITIALIZING BIOTECH DATA PIPELINE ---")


end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')
print(f"Time Window: {start_date} to {end_date}")


print("\nFetching Google Trends RSV for 'Asthma' in Delhi...")
pytrends = TrendReq(hl='en-IN', tz=330)
try:
    pytrends.build_payload(["asthma"], timeframe='today 3-m', geo='IN-DL')
    trends_df = pytrends.interest_over_time()
    if 'isPartial' in trends_df.columns:
        trends_df = trends_df.drop(columns=['isPartial'])
    print(f"Successfully fetched {len(trends_df)} days of RSV data.")
except Exception as e:
    print("Google API Rate Limited. Please run backup cache.")

print("\nFetching Environmental Baseline from Open-Meteo...")
lat, lon = 28.6139, 77.2090
url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=pm2_5,us_aqi&timezone=Asia%2FCalcutta&start_date={start_date}&end_date={end_date}"

response = requests.get(url)
data = response.json()
aqi_df = pd.DataFrame(data['hourly']).dropna()
print(f"Successfully fetched {len(aqi_df)} hours of Air Quality data.")

print("\n--- PIPELINE EXECUTION COMPLETE ---")